# Прогноз из 10:30
Три последовательных недельных окна tuning с расширяющимся train. Каждой неделе предшествуют семь дней val для early stopping. Выбор моделей и калибровки использует прогнозы трёх недель. Последние 14 дат ранее просматривались при разработке. Их локальная оценка получена до финального переобучения.


In [ ]:
from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
DATA = Path('data')
TRAIN_FILE = 'train_solo_track.parquet'
TEST_FILE = 'test_solo_track.parquet'
TRAIN_PATH, TEST_PATH = (DATA / TRAIN_FILE, DATA / TEST_FILE)
if not (TRAIN_PATH.is_file() and TEST_PATH.is_file()) and Path('/kaggle/input').exists():
    train_matches = list(Path('/kaggle/input').rglob(TRAIN_FILE))
    test_matches = list(Path('/kaggle/input').rglob(TEST_FILE))
    if len(train_matches) != 1 or len(test_matches) != 1:
        raise ValueError(f'Укажите TRAIN_PATH и TEST_PATH вручную: train={train_matches}, test={test_matches}')
    TRAIN_PATH, TEST_PATH = (train_matches[0], test_matches[0])
OUT = Path('artifacts/features_1030_windows_v4/features')
MODEL_DIR = OUT.parent / 'models'
IMPORTANCE_DIR = OUT.parent / 'importance'
POSTPROCESS_DIR = OUT.parent / 'postprocessing'
for directory in (OUT, MODEL_DIR, IMPORTANCE_DIR, POSTPROCESS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TARGET_SCALE = 100000
MAX_TREES = 4000
EARLY_STOPPING_ROUNDS = 150
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
PERMUTATION_ENABLED = True
PERMUTATION_TOP_N = 10
PERMUTATION_REPEATS = 3
PERMUTATION_MAX_DAYS = 7
FEATURE_PIPELINE_VERSION = '1030_windows_v4'


## Данные и временные границы
Проверяется полная получасовая панель всех маршрутов. Границы задаются до расчёта любых статистик, использующих цель. Словарь корреляционных лагов обучается только до границы train.


In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if not not history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 28:
    raise ValueError('Недостаточно дат для временного разбиения')
if len(dates) < 63:
    raise ValueError('Недостаточно истории для трёх недель подбора')
holdout_start = dates[-14]
folds = []
for start in (len(dates)-35, len(dates)-28, len(dates)-21):
    folds.append(dict(train=dates[:start-7], val=dates[start-7:start], tuning=dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))


## Исторические признаки и окна
Сохраняются признаки присланного ноутбука: уровни цели и статусов, средние за 2/6 часов, изменения за час, статусные суммы и отношения, суточные лаги, отклонение от среднего и отношение текущего уровня к предыдущим дням.

Вокруг каждого лага строится окно ±4 получасовых шага. Для последних значений и сглаженных сигналов доступны только точки не позже собственного старта: положительные сдвиги не создаются. Одинаковые источники не дублируются: например, соседние окна target_back_0/1/2/4 объединяются в target_back_0…8. Для status_latest добавляются back_1…4. Лаги сутки/двое назад и сезонные слоты получают все девять соседей. Исходные центральные признаки сохранены. Эквивалентные change_1h статусов 4–6 заменены на diff_hourly.

Суммы статусов не являются числом уникальных товаров. Названия warehouse_total_load, hub_buffer и hub_gate_bottleneck обозначают формулы, а не подтверждённые физические показатели. Недоступная история остаётся NaN.

Для цели и каждого статуса рассчитываются общая сумма по всем маршрутам в доступный момент и её изменение за час. Соседние исторические значения используют только левую часть окна у старта.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)


## Корреляционный лаг маршрута
На полной получасовой истории train ищется максимальная положительная Pearson-корреляция target(u) с diff_status6(u−L), L=2…48 шагов. Минимум 336 пар и корреляция >0,05 — эвристики, не тест значимости.

Выбранный L фиксирован для маршрута. Для горизонта h нужен сигнал origin+h−L. Помимо центра добавляются соседи ±2 часа, каждый доступен только при собственном источнике ≤origin. У каждого соседа есть флаг доступности. При слабой связи лаг и сигнал NaN. Для финального refit словарь пересчитывается на всей истории; оценочные parquet и модели сохраняются отдельно.

Для выбора лага и его применения из цели и изменения статуса 6 вычитается среднее того же получаса за предыдущие семь суток. Для каждого окна словарь строится заново до начала val.
Исходный лаг статуса 6 и лаг по остаткам сохраняются отдельными группами.


In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

validation_lag_table = fit_lags(LAG_VALIDATION_FIT_BEFORE)
validation_lag_table.to_csv(OUT.parent / 'status6_lags_validation.csv', index=False)


## Матрицы восьми горизонтов
Каждая историческая строка — маршрут и старт в 10:30 с известной меткой данного горизонта. Сезонные значения берутся относительно прогнозируемого слота, включая ±2 часа. Средние slot_daily7/slot_weekly4 получают отдельное значение для каждого сдвига, а не усреднение внутри окна. Центральные profile28 и overlap используются только постобработкой. Календарь относится к будущему слоту.

Для профилей slot_daily7 и slot_weekly4 вычисляются сдвиг пика в минутах, размах и разность средних правой и левой частей окна. При нескольких равных максимумах используется их средняя позиция. Для календаря остаются день недели, выходной и циклические координаты дня недели.


In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    available_origins = np.flatnonzero((times.hour == 10) & (times.minute == 30) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    frame = horizon_dataset(h, available_origins)
    frame['y'] = target[available_origins + h].ravel()
    if not frame.columns.is_unique or not ((frame.cutoff.dt.hour == 10) & (frame.cutoff.dt.minute == 30)).all():
        raise ValueError('Нарушена схема признаков или время старта')
    frame.to_parquet(path, index=False, compression='zstd')
    features = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp.eq(end + h * STEP)]
    future = requests.merge(features, on=['route_id', 'timestamp'], how='left', validate='one_to_one')
    if not future.cutoff.eq(end).all() or not future.id.equals(requests.id.reset_index(drop=True)):
        raise ValueError('Ошибка сопоставления тестовых строк')
    future.to_parquet(OUT / f'submission_h{h}.parquet', index=False)
    print(f'h={h}: история {len(frame):,}, submission {len(future):,}, модельных признаков {len(frame.columns) - 6}')
for h in range(1, 9):
    write_horizon(h)
feature_names = list(dict.fromkeys((name for h in range(1, 9) for name in pq.read_schema(OUT / f'history_h{h}.parquet').names)))
FEATURE_GROUPS = {'calendar': ['dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'))) and 'lag_48' not in name], 'hub_combinations': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'))) and 'lag_48' not in name], 'historical_lags': [name for name in feature_names if 'lag_48' in name or 'lag_96' in name], 'target_change': [name for name in feature_names if name.startswith('target_acceleration')], 'correlation_lag': [name for name in feature_names if name in CORRELATION_COLUMNS or name.startswith('lagged_status6_signal')], 'seasonal_windows': [name for name in feature_names if name.startswith('slot_')], 'recent_target': [name for name in feature_names if name.startswith('target_back_')], 'recent_statuses': [name for name in feature_names if any((name.startswith(f'status_{i}_back_') or name == f'status_{i}_latest' for i in range(1, 7)))]}

del values, means, level24, extra_signals, positions, recent, usual
FEATURE_GROUPS['network'] = [name for name in feature_names if name.startswith('network_')]
FEATURE_GROUPS['residual_correlation_lag'] = [name for name in feature_names if name.startswith('residual_')]
(OUT.parent / 'feature_pipeline.json').write_text(json.dumps({'version': FEATURE_PIPELINE_VERSION, 'window_offsets_steps': list(WINDOW_OFFSETS), 'training_origin': '10:30', 'validation_lag_fit_before': str(LAG_VALIDATION_FIT_BEFORE), 'splits': {key: [str(period.min()), str(period.max())] for key, period in split_dates.items()}, 'feature_groups': FEATURE_GROUPS}, ensure_ascii=False, indent=2))


## Подбор по трём последовательным неделям
Для каждой конфигурации собираются прогнозы tuning трёх моделей, каждая использует только более ранние train и val. Победитель выбирается по среднему дневному score за 21 дату. Модель последнего окна даёт локальный test и число деревьев для refit.


In [ ]:
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
if get_gpu_device_count() < 1:
    raise RuntimeError('Для обучения включите GPU в настройках Kaggle Notebook.')
MODEL_DIR = OUT.parent / 'models'
MODEL_DIR.mkdir(exist_ok=True)
print('Начало подбора:', tune_start, 'Начало отложенной проверки:', holdout_start)

def make_model(trees, params):
    return CatBoostRegressor(iterations=trees, **params, loss_function='MAE', eval_metric='MAE', task_type='GPU', devices='0', cat_features=['route_id'], random_seed=42, allow_writing_files=False, verbose=200)

def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()
IMPORTANCE_DIR = OUT.parent / 'importance'
IMPORTANCE_DIR.mkdir(exist_ok=True)
builtin_importance_parts = []

def record_builtin_importance(model, columns, h, stage):
    importance = model.get_feature_importance(type='PredictionValuesChange')
    if len(importance) != len(columns):
        raise ValueError('Важности не совпадают с матрицей признаков')
    builtin_importance_parts.append(pd.DataFrame({'feature': columns, 'importance': importance, 'horizon': h, 'stage': stage}))
SEARCH_CONFIGS = [{'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'bootstrap_type': 'No'}, {'depth': 6, 'learning_rate': 0.07, 'l2_leaf_reg': 10, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1}, {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'bootstrap_type': 'No'}, {'depth': 8, 'learning_rate': 0.05, 'l2_leaf_reg': 10, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1}, {'depth': 10, 'learning_rate': 0.03, 'l2_leaf_reg': 10, 'bootstrap_type': 'No'}, {'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 3, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.5}]


In [ ]:
selected = {}
evaluation_tables = {}
validation_parts = []
search_rows = []
transfer_parts = {1: [], 2: []}
lag_tables = [fit_lags(f['val'][0].normalize()) for f in folds[:-1]] + [validation_lag_table]
for h in range(1,9):
    history_frame = pd.read_parquet(OUT / f'history_h{h}.parquet')
    columns = model_columns(history_frame)
    fold_frames = [add_correlation_features(history_frame,h,t) for t in lag_tables]
    scores = []
    prefix_scores = {1: np.inf, 2: np.inf}
    prefix_panels = {}
    for trial, params in enumerate(SEARCH_CONFIGS,1):
        parts = []
        for index, (fold, data) in enumerate(zip(folds, fold_frames)):
            masks = {name: data.cutoff.isin(days) for name,days in fold.items()}
            model = make_model(MAX_TREES, params)
            model.fit(data.loc[masks['train'], columns], data.loc[masks['train'],'y']/TARGET_SCALE,
                      eval_set=(data.loc[masks['val'],columns], data.loc[masks['val'],'y']/TARGET_SCALE),
                      early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True)
            rows = []
            for split,mask in masks.items():
                subset = data.loc[mask]
                pred = np.maximum(model.predict(subset[columns])*TARGET_SCALE,0)
                rows.append(score_row(subset,pred,split))
                if split == 'tuning' or (index == len(folds)-1 and split == 'test'):
                    meta = subset[['cutoff','timestamp','route_id','y','profile28','overlap']].copy()
                    meta['catboost'] = pred
                    meta['seasonal'] = subset[['slot_daily7','slot_weekly4']].mean(axis=1)
                    meta['horizon'] = h
                    meta['period'] = split
                    parts.append(meta)
            table = pd.DataFrame(rows)
            print(f'h={h}, конфигурация {trial}, окно {index+1}, деревьев {model.tree_count_}')
            display(table)
            search_rows.extend([dict(horizon=h,trial=trial,fold=index+1,trees=model.tree_count_,**row) for row in rows])
        combined = pd.concat(parts,ignore_index=True)
        oos = combined.loc[combined.period.eq('tuning')]
        score = mean_daily_score(oos,oos.catboost)
        if not np.isfinite(score):
            raise ValueError("Неконечная метрика подбора")
        for stop in transfer_parts:
            train_days = folds[stop]['tuning'][0]
            earlier = oos.loc[oos.cutoff < train_days]
            prefix_score = mean_daily_score(earlier, earlier.catboost)
            if prefix_score < prefix_scores[stop]:
                prefix_scores[stop] = prefix_score
                panel = oos.loc[oos.cutoff <= folds[stop]['tuning'][-1]].copy()
                panel['trial'] = trial
                prefix_panels[stop] = panel
        if not scores or score < min(scores):
            selected[h] = dict(params=params.copy(),trees=int(model.tree_count_),trial=trial,tuning_score=float(score))
            evaluation_tables[h] = table.copy()
            best_part = combined
            model.save_model(str(MODEL_DIR / f'catboost_h{h}_validation.cbm'))
        scores.append(score)
    winner = CatBoostRegressor()
    winner.load_model(str(MODEL_DIR / f'catboost_h{h}_validation.cbm'))
    record_builtin_importance(winner,columns,h,'validation_model')
    validation_parts.append(best_part)
    for stop in transfer_parts:
        transfer_parts[stop].append(prefix_panels[stop])
    pd.DataFrame(search_rows).to_csv(OUT.parent/'hyperparameter_search.csv',index=False)
    (OUT.parent/'selected_models.json').write_text(json.dumps(selected,indent=2))
validation = pd.concat(validation_parts,ignore_index=True)
validation.to_parquet(OUT.parent/'validation_predictions.parquet',index=False)
print('Локальный test оценён до финального переобучения. Эти даты уже использовались при разработке.')


## Перенос калибровки на следующую неделю
В каждом окне конфигурация CatBoost для каждого горизонта выбирается только по предшествующим неделям tuning. Сравниваются полные варианты постобработки: для каждого семейства на train подбираются веса смеси, overlap, профиль и параметры поправки. Val получает зафиксированные настройки. Train и val здесь обозначают периоды проверки калибровки, отдельные от обучающих строк CatBoost и его early stopping.

Побеждает семейство с наименьшим средним дневным WAPE + |Relative Bias| на двух следующих неделях. Затем его смесь и параметры оцениваются на всех 21 днях tuning. Локальный test служит для последующей оценки выбранного решения.

Аффинная поправка использует неотрицательный наклон и свободный член. Для горизонтов 1, 2–4 и 5–8 множители по объёмам стягиваются к общему коэффициенту. Нелинейная кривая обучается изотонической регрессией по восьми квантильным интервалам прогноза. За крайними узлами применяется значение ближайшего узла. Все параметры подбираются на train окна.


In [ ]:
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize_scalar

def get_raw_full_prediction(frame, weight, use_overlap, profile_weight):
    base = weight * frame.catboost.to_numpy(dtype=np.float64) + (1 - weight) * frame.seasonal.to_numpy(dtype=np.float64)
    first = frame.horizon.to_numpy() == 1
    if use_overlap:
        base[first] = frame.loc[first, 'overlap'].to_numpy()
    later = ~first
    base[later] = (1 - profile_weight) * base[later] + profile_weight * frame.loc[later, 'profile28'].to_numpy()
    return base

def apply_final_calibration(prediction, kind, value, horizon=None):
    prediction = np.asarray(prediction, dtype=np.float64)
    if kind == 'none':
        result = prediction
    elif kind == 'multiplicative':
        result = prediction * value
    elif kind == 'affine':
        result = prediction * value['k'] + value['b']
    elif kind == 'grouped':
        groups = np.select([np.asarray(horizon)==1, np.asarray(horizon)<=4], [0,1], default=2)
        result = prediction * np.asarray(value['factors'])[groups]
    elif kind == 'nonlinear':
        result = np.interp(prediction, value['x'], value['y'])
    elif kind == 'additive':
        result = prediction + value
    else:
        raise ValueError(f'Неизвестная калибровка: {kind}')
    return np.maximum(result, 0)

def calibration_objective(frame, raw):
    actual = frame.y.to_numpy(dtype=np.float64)
    day_codes, unique_days = pd.factorize(frame.cutoff.dt.normalize(), sort=True)
    volumes = np.bincount(day_codes, weights=actual, minlength=len(unique_days))
    valid = volumes > 0
    if not valid.any() or not np.isfinite(raw).all():
        raise ValueError('Некорректные данные для калибровки')

    def objective(kind, value):
        error = apply_final_calibration(raw, kind, value, frame.horizon) - actual
        absolute = np.bincount(day_codes, weights=np.abs(error), minlength=len(unique_days))
        signed = np.bincount(day_codes, weights=error, minlength=len(unique_days))
        return float(np.mean((absolute[valid] + np.abs(signed[valid])) / volumes[valid]))
    return objective

def fit_calibration_value(frame, raw, kind):
    raw = np.asarray(raw, dtype=np.float64)
    objective = calibration_objective(frame, raw)
    if kind == 'affine':
        scale = max(float(frame.y.mean()), 1.0)
        def params(x):
            return dict(k=float(x[0]), b=float(x[1]*scale))
        initial = np.array([1., 0.])
        fit = minimize(lambda x: objective(kind,params(x)), initial, method='Powell',
                       bounds=[(0.,3.),(-2.,2.)], options={'maxiter':40})
        return min([params(initial),params(fit.x)],key=lambda v: objective(kind,v))
    if kind == 'grouped':
        h = frame.horizon.to_numpy()
        factors = []
        total = raw.sum()
        common = frame.y.sum()/total if total>0 else 1.0
        for mask in (h==1, (h>=2)&(h<=4), h>=5):
            volume = raw[mask].sum()
            ratio = frame.y.to_numpy()[mask].sum()/volume if volume>0 else common
            factors.append(float(np.clip(ratio,0,3)))
        options = [dict(factors=np.clip(common + strength*(np.array(factors)-common),0,3).tolist())
                   for strength in (0.,0.25,0.5,0.75,1.)]
        options.append(dict(factors=[1.,1.,1.]))
        return min(options,key=lambda v: objective(kind,v))
    if kind == 'nonlinear':
        cuts = np.unique(np.quantile(raw,np.linspace(0,1,9)[1:-1]))
        bins = np.searchsorted(cuts,raw)
        weights = 1/frame.groupby('cutoff').y.transform('sum').replace(0,np.inf).to_numpy()
        counts = np.bincount(bins,weights=weights)
        keep = counts>0
        x = np.bincount(bins,weights=weights*raw)[keep]/counts[keep]
        y = np.bincount(bins,weights=weights*frame.y.to_numpy())[keep]/counts[keep]
        curve = IsotonicRegression().fit_transform(x,y,sample_weight=counts[keep])
        return dict(x=x.tolist(),y=curve.tolist())
    if kind == 'none':
        return 0.0
    if kind == 'multiplicative':
        lower, upper = (0.0, 3.0)
        bias_value = frame.y.sum() / raw.sum() if raw.sum() > 0 else 1.0
        candidates = [lower, upper, 1.0, float(np.clip(bias_value, lower, upper))]
    elif kind == 'additive':
        scale = max(float(frame.y.mean()), 1.0)
        lower, upper = (-2 * scale, 2 * scale)
        candidates = [lower, upper, 0.0, float(np.clip(np.mean(frame.y.to_numpy() - raw), lower, upper))]
    else:
        raise ValueError(f'Неизвестная калибровка: {kind}')
    grid = np.linspace(lower, upper, 65)
    grid_scores = np.array([objective(kind,x) for x in grid])
    candidates.append(float(grid[grid_scores.argmin()]))
    minima = [i for i in range(len(grid)) if grid_scores[i] <= grid_scores[max(i-1,0)] and grid_scores[i] <= grid_scores[min(i+1,len(grid)-1)]]
    for i in sorted(minima,key=lambda i:grid_scores[i])[:5]:
        interval = (grid[max(i-1,0)], grid[min(i+1,len(grid)-1)])
        result = minimize_scalar(lambda x: objective(kind,x), bounds=interval, method='bounded')
        if result.success:
            candidates.append(float(result.x))
    return min(candidates, key=lambda value: objective(kind, value))

KINDS = ('none','multiplicative','additive','affine','grouped','nonlinear')

def search_calibrations(frame, kinds):
    best = {}
    rows = []
    for weight in np.linspace(0,1,11):
        for overlap in (False, True):
            for profile in np.linspace(0,1,11):
                raw = get_raw_full_prediction(frame,weight,overlap,profile)
                objective = calibration_objective(frame,raw)
                for kind in kinds:
                    value = fit_calibration_value(frame,raw,kind)
                    score = objective(kind,value)
                    config = dict(w=float(weight),overlap=overlap,p_weight=float(profile),
                                  calib_type=kind,calib_val=value,tuning_score=float(score))
                    rows.append(config)
                    if kind not in best or score < best[kind]['tuning_score']:
                        best[kind] = config
    return best, pd.DataFrame(rows)

blend_oos = []
transfer_rows = []
transfer_settings = []
for stop, parts in transfer_parts.items():
    panel = pd.concat(parts,ignore_index=True)
    boundary = folds[stop]['tuning'][0]
    train = panel.loc[panel.cutoff < boundary]
    val = panel.loc[panel.cutoff >= boundary]
    if train.cutoff.max() >= val.cutoff.min() or val.cutoff.nunique()!=7:
        raise ValueError('Нарушен порядок недель калибровки')
    configs, _ = search_calibrations(train,KINDS)
    prefix_choice = min(configs.values(), key=lambda config: config['tuning_score'])
    out = val[['cutoff','timestamp','route_id','horizon','y']].copy()
    raw = get_raw_full_prediction(val,prefix_choice['w'],prefix_choice['overlap'],prefix_choice['p_weight'])
    out['fe_prediction'] = apply_final_calibration(raw,prefix_choice['calib_type'],prefix_choice['calib_val'],val.horizon)
    out['model_info_through'] = train.timestamp.max()
    blend_oos.append(out)
    for kind, config in configs.items():
        transfer_settings.append(dict(window=stop, models=panel.groupby('horizon').trial.first().to_dict(), **config))
        for label, subset in [('train',train),('val',val)]:
            raw = get_raw_full_prediction(subset,config['w'],config['overlap'],config['p_weight'])
            pred = apply_final_calibration(raw,kind,config['calib_val'],subset.horizon)
            raw_score = mean_daily_score(subset,raw)
            transfer_rows.append(dict(window=stop, family=kind, first=subset.cutoff.min(),
                                      last=subset.cutoff.max(), raw_score=raw_score, gain_vs_same_mix=raw_score-mean_daily_score(subset,pred), **score_row(subset,pred,label)))
pd.concat(blend_oos,ignore_index=True).to_parquet(POSTPROCESS_DIR/'fe_forward_predictions.parquet',index=False)
transfer_metrics = pd.DataFrame(transfer_rows)
transfer_metrics.to_csv(POSTPROCESS_DIR/'calibration_transfer.csv',index=False)
display(transfer_metrics)
transfer_summary = transfer_metrics.pivot_table(index='family',columns='выборка',values='среднее по дням').sort_values('val')
transfer_summary.to_csv(POSTPROCESS_DIR/'calibration_transfer_summary.csv')
(POSTPROCESS_DIR/'calibration_transfer_settings.json').write_text(json.dumps(transfer_settings,ensure_ascii=False,indent=2))
display(transfer_summary)
chosen_family = transfer_summary.index[0]
tuning = validation.loc[validation.period.eq('tuning')].copy()
holdout = validation.loc[validation.period.eq('test')].copy()
configs, postprocess_search = search_calibrations(tuning,[chosen_family])
best_config = configs[chosen_family].copy()
best_score = best_config.pop('tuning_score')
submission_calib_val = best_config['calib_val']
postprocess_search.to_csv(POSTPROCESS_DIR/'search.csv',index=False)
print('Выбрано по переносу на val:',best_config)


## Проверка комбинации и отчёт по ошибкам
Test оценён до переобучения. Эти даты уже использовались в проекте.

Ошибки откалиброванной смеси группируются по горизонту, дню недели, квартилю объёма маршрута на раннем train и нулевому факту. У группы с нулевой суммой факта WAPE не определён. Её вклад виден по сумме абсолютных ошибок.


In [ ]:
postprocess_metrics = []
validation_postprocessed = validation.copy()
raw_val_pred = get_raw_full_prediction(validation, best_config['w'], best_config['overlap'], best_config['p_weight'])
validation_postprocessed['raw_full_prediction'] = raw_val_pred
validation_postprocessed['y_pred_tuning_calibration'] = apply_final_calibration(raw_val_pred, best_config['calib_type'], submission_calib_val, validation.horizon)
validation_postprocessed.to_parquet(POSTPROCESS_DIR / 'validation_postprocessed.parquet', index=False)
for split, subset in (('tuning', tuning), ('test', holdout)):
    raw = get_raw_full_prediction(subset, best_config['w'], best_config['overlap'], best_config['p_weight'])
    for label, prediction in (('сезонный', subset.seasonal), ('CatBoost', subset.catboost), ('комбинация без калибровки', raw), ('комбинация с tuning-калибровкой', apply_final_calibration(raw, best_config['calib_type'], submission_calib_val, subset.horizon))):
        postprocess_metrics.append({'period': split, **score_row(subset, prediction, label)})
postprocess_metrics = pd.DataFrame(postprocess_metrics)
display(postprocess_metrics)
postprocess_metrics.to_csv(POSTPROCESS_DIR / 'metrics.csv', index=False)
postprocess_metrics.loc[postprocess_metrics.period.eq('test')].to_csv(POSTPROCESS_DIR / 'holdout_metrics.csv', index=False)
diagnostic_rows = []
for label, use_overlap, profile_weight in (('исходная смесь', False, 0.0), ('только замена h1', True, 0.0), ('только профиль', False, best_config['p_weight']), ('выбранная комбинация', best_config['overlap'], best_config['p_weight'])):
    raw = get_raw_full_prediction(tuning, best_config['w'], use_overlap, profile_weight)
    value = fit_calibration_value(tuning, raw, best_config['calib_type'])
    row = {'вариант': label, 'вес профиля': profile_weight}
    for period, subset in (('tuning', tuning), ('test', holdout), ('субботы test', holdout.loc[holdout.cutoff.dt.dayofweek.eq(5)])):
        if subset.empty:
            row[period] = np.nan
        else:
            raw = get_raw_full_prediction(subset, best_config['w'], use_overlap, profile_weight)
            row[period] = mean_daily_score(subset, apply_final_calibration(raw, best_config['calib_type'], value, subset.horizon))
    diagnostic_rows.append(row)
postprocessing_diagnostics = pd.DataFrame(diagnostic_rows)
display(postprocessing_diagnostics)
postprocessing_diagnostics.to_csv(POSTPROCESS_DIR / 'overlap_profile_diagnostics.csv', index=False)
(POSTPROCESS_DIR / 'settings.json').write_text(json.dumps({'best_config': best_config, 'tuning_score': best_score, 'calibration_source': 'tuning_only_unchanged_for_submission', 'test_evaluated_before_refit': True, 'target_scale': TARGET_SCALE}, ensure_ascii=False, indent=2))

errors = validation_postprocessed.loc[validation_postprocessed.period.eq('test')].copy()
errors['weekday'] = errors.cutoff.dt.dayofweek
errors['zero_actual'] = errors.y.eq(0)
route_means = pd.Series(target[times < folds[0]['val'][0].normalize()].mean(axis=0),index=routes)
size = pd.qcut(route_means.rank(method='first'),4,labels=False,duplicates='drop')
errors['route_size_quartile'] = errors.route_id.map(size)
detail = []
for dimension in ['horizon','weekday','zero_actual','route_size_quartile']:
    for key,group in errors.groupby(dimension):
        residual = group.y_pred_tuning_calibration - group.y
        detail.append(dict(dimension=dimension,group=str(key),absolute_error=residual.abs().sum(),signed_error=residual.sum(),**score_row(group,group.y_pred_tuning_calibration,'test')))
error_report = pd.DataFrame(detail)
display(error_report)
error_report.to_csv(POSTPROCESS_DIR/'test_error_report.csv',index=False)


## Финальное обучение на всей размеченной истории со стартами 10:30
Словарь лагов пересчитывается на полной получасовой истории. Обучение каждой лучшей конфигурации использует все дни 10:30 с известной меткой своего горизонта, включая бывшие tuning, val и test. Число деревьев фиксировано предыдущим этапом. Сохранённые оценки train/test относятся к моделям до refit. Коэффициент калибровки остаётся с tuning; перенос его на переобученную модель — допущение, а не гарантия сохранения bias.

Оценка test независима от финального переобучения и получена до него. При интерпретации учитывайте предыдущие эксперименты на этих датах.


In [ ]:
final_lag_table = fit_lags(end + STEP)
final_lag_table.to_csv(OUT.parent / 'status6_lags_final.csv', index=False)
metadata_path = OUT.parent / 'feature_pipeline.json'
metadata = json.loads(metadata_path.read_text())
metadata.update(final_lag_fit_before=str(end + STEP), target_scale=TARGET_SCALE)
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
submission_parts = []
for h in range(1, 9):
    data = add_correlation_features(pd.read_parquet(OUT / f'history_h{h}.parquet'), h, final_lag_table)
    future = add_correlation_features(pd.read_parquet(OUT / f'submission_h{h}.parquet'), h, final_lag_table)
    columns = model_columns(data)
    if not ((data.cutoff.dt.hour == 10) & (data.cutoff.dt.minute == 30)).all():
        raise ValueError('Финальный train содержит старты не в 10:30')
    model = make_model(selected[h]['trees'], selected[h]['params'])
    model.fit(data[columns], data.y / TARGET_SCALE)
    model.save_model(str(MODEL_DIR / f'catboost_h{h}_scaled.cbm'))
    record_builtin_importance(model, columns, h, 'final_model')
    result = future[['id', 'profile28', 'overlap']].copy()
    result['seasonal'] = future[['slot_daily7', 'slot_weekly4']].mean(axis=1)
    result['horizon'] = h
    result['catboost'] = np.maximum(model.predict(future[columns]) * TARGET_SCALE, 0)
    full_pred = np.maximum(model.predict(data[columns]) * TARGET_SCALE, 0)
    full_row = score_row(data, full_pred, 'вся история 10:30 после переобучения')
    print(f'h={h}: локальная оценка ранее просмотренного test получена до переобучения; последняя строка — in-sample')
    display(pd.concat([evaluation_tables[h], pd.DataFrame([full_row])], ignore_index=True))
    submission_parts.append(result)
    del model, data, future, full_pred
    gc.collect()


## Submission
Комбинация и калибровка применяются ровно один раз после объединения горизонтов. Параметры и коэффициент взяты только с tuning. Порядок id сохраняется из конкурсного test.


In [ ]:
predictions = pd.concat(submission_parts, ignore_index=True)
raw_submission_pred = get_raw_full_prediction(predictions, best_config['w'], best_config['overlap'], best_config['p_weight'])
predictions['y_pred'] = apply_final_calibration(raw_submission_pred, best_config['calib_type'], submission_calib_val, predictions.horizon)
submission = test[['id']].merge(predictions[['id', 'y_pred']], on='id', how='left', validate='one_to_one')
if len(predictions) != len(test) or len(submission) != len(test) or (not predictions.id.is_unique) or (set(predictions.horizon.unique()) != set(range(1, 9))) or (not submission.id.equals(test.id.reset_index(drop=True))) or (not np.isfinite(submission.y_pred).all()) or (submission.y_pred < 0).any():
    raise ValueError('Неполный или некорректный submission')
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(submission):,} строк. Локальная оценка ранее просмотренного test получена до финального переобучения.')
display(submission.head())


## Важности CatBoost
PredictionValuesChange сохраняется отдельно для победителей до refit и финальных моделей. Таблица и график содержат средние важности по восьми горизонтам и разбивку по горизонтам. Важность не является причинным эффектом.


In [ ]:
import matplotlib.pyplot as plt
builtin_importance = pd.concat(builtin_importance_parts, ignore_index=True)
builtin_importance.to_csv(IMPORTANCE_DIR / 'catboost_importance_by_horizon.csv', index=False)
importance_summary = builtin_importance.groupby(['stage', 'feature']).importance.agg(['mean', 'std']).reset_index().sort_values(['stage', 'mean'], ascending=[True, False])
importance_summary.to_csv(IMPORTANCE_DIR / 'catboost_importance_summary.csv', index=False)
display(importance_summary.groupby('stage', sort=False).head(30))
importance_by_horizon = builtin_importance.query("stage == 'final_model'").pivot(index='feature', columns='horizon', values='importance')
importance_by_horizon = importance_by_horizon.loc[importance_by_horizon.mean(axis=1).sort_values(ascending=False).index]
display(importance_by_horizon.head(30))
top = importance_summary.query("stage == 'final_model'").head(30).sort_values('mean')
fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(top.feature, top['mean'], xerr=top['std'].fillna(0))
ax.set(xlabel='Средняя PredictionValuesChange', title='Финальные модели: top-30')
fig.tight_layout()
fig.savefig(IMPORTANCE_DIR / 'catboost_top30.png', dpi=150, bbox_inches='tight')
plt.show()


## Блочная permutation importance на test моделями до refit
Используются сохранённые оценочные модели и исходный train-словарь лагов. По умолчанию 7 дат test, все маршруты, 3 повтора, top-10 признаков, центральный корреляционный сигнал и его лаг, а также группы новых признаков и окон. Весь сезонный блок переставляется вместе, чтобы увидеть зависимость при взаимозаменяемых соседях.

Переставляются маршрутные траектории с циклическим сдвигом дат. Один индекс применяется к группе и всем горизонтам. Положительное изменение среднего дневного WAPE + |bias| означает ухудшение. horizon=0 объединяет все маршруты и горизонты дня. Это диагностика чистого CatBoost, без смеси и калибровки, без нового обучения или отбора. После использования test-важностей для нового подбора потребуется новая внешняя проверка.


In [ ]:
def block_permutation_index(frame, seed):
    rng = np.random.default_rng(seed)
    days = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    route_values = np.sort(frame.route_id.unique())
    keys = pd.MultiIndex.from_frame(frame[['cutoff', 'route_id']])
    if not keys.is_unique or len(frame) != len(days) * len(route_values):
        raise ValueError('Permutation importance требует полную панель дата × маршрут')
    date_shift = int(rng.integers(1, len(days))) if len(days) > 1 else 0
    donor_days = days[np.roll(np.arange(len(days)), date_shift)][days.get_indexer(frame.cutoff)]
    donor_routes = rng.permutation(route_values)[pd.Index(route_values).get_indexer(frame.route_id)]
    donor = keys.get_indexer(pd.MultiIndex.from_arrays([donor_days, donor_routes]))
    if (donor < 0).any() or len(np.unique(donor)) != len(frame):
        raise ValueError('Перестановка не является биекцией')
    return donor

def permutation_deltas(meta_parts, original_parts, shuffled_parts):
    rows = []
    for h, meta, original, shuffled in zip(range(1, 9), meta_parts, original_parts, shuffled_parts):
        before, after = (score_row(meta, original, 'baseline'), score_row(meta, shuffled, 'permuted'))
        rows.append({'horizon': h, 'delta_mean_daily_score': after['среднее по дням'] - before['среднее по дням'], 'delta_pooled_score': after['метрика конкурса'] - before['метрика конкурса'], 'delta_wape': after['WAPE'] - before['WAPE'], 'delta_abs_bias': after['|Relative Bias|'] - before['|Relative Bias|']})
    combined = pd.concat(meta_parts, ignore_index=True)
    before = score_row(combined, np.concatenate(original_parts), 'baseline')
    after = score_row(combined, np.concatenate(shuffled_parts), 'permuted')
    rows.append({'horizon': 0, 'delta_mean_daily_score': after['среднее по дням'] - before['среднее по дням'], 'delta_pooled_score': after['метрика конкурса'] - before['метрика конкурса'], 'delta_wape': after['WAPE'] - before['WAPE'], 'delta_abs_bias': after['|Relative Bias|'] - before['|Relative Bias|']})
    return rows

def run_permutation_importance():
    if PERMUTATION_MAX_DAYS < 2 or PERMUTATION_REPEATS < 2 or PERMUTATION_TOP_N < 1:
        raise ValueError('Для диагностики нужны >=2 даты, >=2 повтора и положительный top-N')
    available_days = dates[dates >= holdout_start]
    pick = np.unique(np.linspace(0, len(available_days) - 1, min(PERMUTATION_MAX_DAYS, len(available_days))).round().astype(int))
    selected_days = available_days[pick]
    models, matrices, metadata, original = ([], [], [], [])
    baseline_rows = []
    for h in range(1, 9):
        model = CatBoostRegressor()
        model.load_model(str(MODEL_DIR / f'catboost_h{h}_validation.cbm'))
        names = model.feature_names_
        read_columns = list(dict.fromkeys(['cutoff', 'timestamp', 'route_id', 'y', *names]))
        frame = pd.read_parquet(OUT / f'history_h{h}.parquet', columns=read_columns, filters=[('cutoff', 'in', selected_days.to_pydatetime().tolist())])
        frame = frame.sort_values(['cutoff', 'route_id']).reset_index(drop=True)
        if len(frame) != len(selected_days) * len(routes):
            raise ValueError('Неполная панель отложенных данных для важностей')
        meta = frame[['cutoff', 'timestamp', 'route_id', 'y']].copy()
        matrix = frame[names].copy()
        prediction = (model.predict(matrix) * TARGET_SCALE).clip(min=0)
        baseline_rows.append({'horizon': h, **score_row(meta, prediction, 'permutation_baseline')})
        models.append(model)
        matrices.append(matrix)
        metadata.append(meta)
        original.append(prediction)
    combined = pd.concat(metadata, ignore_index=True)
    baseline_rows.append({'horizon': 0, **score_row(combined, np.concatenate(original), 'permutation_baseline')})
    pd.DataFrame(baseline_rows).to_csv(IMPORTANCE_DIR / 'permutation_baseline.csv', index=False)
    rank = builtin_importance.query("stage == 'validation_model'").groupby('feature').importance.mean().sort_values(ascending=False)
    individual = list(dict.fromkeys([*rank.head(PERMUTATION_TOP_N).index, 'lagged_status6_signal', 'selected_status6_lag_steps']))
    available_columns = set().union(*(set(matrix.columns) for matrix in matrices))
    specs = [('feature', name, [name]) for name in individual if name in available_columns]
    specs += [('group', name, [c for c in names if c in available_columns]) for name, names in FEATURE_GROUPS.items()]
    specs = [spec for spec in specs if spec[2]]
    (IMPORTANCE_DIR / 'permutation_settings.json').write_text(json.dumps({'days': [str(d) for d in selected_days], 'routes': len(routes), 'repeats': PERMUTATION_REPEATS, 'seed': 42, 'model_stage': 'validation_model', 'lag_fit_before': str(LAG_VALIDATION_FIT_BEFORE), 'specs': [{'kind': kind, 'name': name, 'columns': columns} for kind, name, columns in specs]}, ensure_ascii=False, indent=2))
    rows = []
    for kind, name, columns in specs:
        for repeat in range(PERMUTATION_REPEATS):
            shuffled = []
            for model, matrix, meta in zip(models, matrices, metadata):
                donor = block_permutation_index(meta, seed=42 + repeat)
                permuted = matrix.copy()
                active_columns = [column for column in columns if column in matrix.columns]
                permuted.loc[:, active_columns] = matrix.iloc[donor][active_columns].to_numpy()
                shuffled.append((model.predict(permuted) * TARGET_SCALE).clip(min=0))
            for row in permutation_deltas(metadata, original, shuffled):
                rows.append({'kind': kind, 'name': name, 'repeat': repeat, 'n_features': len(columns), **row})
        print(f'Permutation importance: {kind} / {name}')
    detail = pd.DataFrame(rows)
    detail.to_csv(IMPORTANCE_DIR / 'permutation_detail.csv', index=False)
    metrics = ['delta_mean_daily_score', 'delta_pooled_score', 'delta_wape', 'delta_abs_bias']
    summary = detail.groupby(['kind', 'name', 'horizon'])[metrics].agg(['mean', 'std'])
    summary.columns = ['_'.join(column) for column in summary.columns]
    summary = summary.reset_index().sort_values('delta_mean_daily_score_mean', ascending=False)
    summary.to_csv(IMPORTANCE_DIR / 'permutation_summary.csv', index=False)
    return summary
if PERMUTATION_ENABLED:
    permutation_summary = run_permutation_importance()
    overall = permutation_summary.loc[permutation_summary.horizon.eq(0)].copy()
    display(overall)
    plot = overall.sort_values('delta_mean_daily_score_mean')
    fig, ax = plt.subplots(figsize=(11, max(5, len(plot) * 0.3)))
    ax.barh(plot.kind + ': ' + plot.name, plot.delta_mean_daily_score_mean, xerr=plot.delta_mean_daily_score_std.fillna(0))
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set(xlabel='Изменение среднего дневного WAPE + |bias|, все 8 горизонтов', ylabel='Признак или группа', title='Отложенные даты: блочная permutation importance, среднее ± std')
    fig.tight_layout()
    fig.savefig(IMPORTANCE_DIR / 'permutation_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Permutation importance отключена настройкой PERMUTATION_ENABLED; важности CatBoost показаны выше.')

importance_final_table = importance_by_horizon.copy()
importance_final_table['mean_all_8'] = importance_final_table.fillna(0).sum(axis=1)/8
importance_final_table = importance_final_table.sort_values('mean_all_8',ascending=False)
importance_final_table.to_csv(IMPORTANCE_DIR/'final_feature_importance_table.csv')
display(importance_final_table)
